# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides users in loading, exploring, and processing the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Examine the dataset's structure to find record sets and their IDs
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets defined directly in metadata. Attempting to identify from schema ...")
    # Try to load from dataset.schema if available
    schema = dataset.schema
    record_sets = [obj["@id"] for obj in schema if obj.get("@type") == "cr:RecordSet"]

print("Record Sets (@id):")
for record_set_id in record_sets:
    print(record_set_id)

# Display fields and columns in each record set
for record_set_id in record_sets:
    print(f"\nFields and columns for record set {record_set_id}:")
    fields = []
    columns = []
    for obj in schema:
        if obj.get("@id") == record_set_id:
            # fields: cr:field
            if "field" in obj:
                field_objs = obj["field"]
                if isinstance(field_objs, list):
                    fields.extend([fld["@id"] if isinstance(fld, dict) else fld for fld in field_objs])
                else:
                    fields.append(field_objs["@id"] if isinstance(field_objs, dict) else field_objs)
            # columns: cr:column
            if "column" in obj:
                col_objs = obj["column"]
                if isinstance(col_objs, list):
                    columns.extend([col["@id"] if isinstance(col, dict) else col for col in col_objs])
                else:
                    columns.append(col_objs["@id"] if isinstance(col_objs, dict) else col_objs)
    print(f"Fields (@id): {fields}")
    print(f"Columns (@id): {columns}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Note_: For demonstration, all record sets found are loaded. Adjust the `record_sets` variable if needed to target specific record sets.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading records from record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id} with columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for this record set.")
    except Exception as e:
        print(f"Unable to load records for {record_set_id}: {e}")

# Select the first record set for demonstration
selected_record_set = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use field/column `@id`s.

In [ ]:
import numpy as np

# Example: Let's attempt EDA using first record set
if selected_record_set and selected_record_set in dataframes:
    df = dataframes[selected_record_set]
    print(f"\nEDA for record set @id: {selected_record_set}")

    # Determine numeric fields (by type or by inspecting) from the schema
    numeric_fields = []
    for obj in schema:
        if obj.get("@type") == "cr:Field" and obj.get("source") == selected_record_set:
            if obj.get("dataType") in ["schema:Float", "schema:Integer", "schema:Number"]:
                numeric_fields.append(obj["@id"])

    # If no numeric fields found, try to infer from dataframe
    if not numeric_fields:
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    print(f"Numeric fields (@id): {numeric_fields}")

    if numeric_fields:
        numeric_field_id = numeric_fields[0]

        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping (choose a non-numeric field if available)
        group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (@id):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: histogram and scatter
if selected_record_set and selected_record_set in dataframes:
    df = dataframes[selected_record_set]
    if numeric_fields:
        # Plot a histogram
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_fields[0]].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {numeric_fields[0]} (@id)")
        plt.xlabel(numeric_fields[0])
        plt.ylabel("Count")
        plt.show()

        if len(numeric_fields) > 1:
            # Scatter plot between first two numeric fields
            plt.figure(figsize=(6,4))
            sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
            plt.title(f"Scatter: {numeric_fields[0]} vs {numeric_fields[1]}")
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.show()
        else:
            print("Only one numeric field available for visualization.")
    else:
        print("No numeric fields found for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze the FAIR² clinical dataset using `mlcroissant` by referencing record sets and fields using their `@id`s.

Key outcomes:
- Loaded dataset metadata and explored its structure
- Identified record sets and their fields/columns by `@id`
- Extracted tabular data and applied basic EDA operations
- Visualized distributions for numeric fields

This workflow can be adapted to larger clinical, imaging, or genomic datasets structured as Croissant schemas. For more advanced analysis or model building, larger and more diverse datasets are recommended.